In [ ]:
import meshio as mio
import h5py
import numpy as np
import pyvista as pv
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import scipy as sp

pv.set_jupyter_backend('trame')


In [ ]:
def to_pyvista_mesh(V, F = None):
    if F is None:
        return pv.PolyData(V)
    if F.shape[1] == 3:
        return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)
    elif F.shape[1] == 4:
        return pv.UnstructuredGrid({pv.CellType.TETRA: F}, V)



In [ ]:
# path = "/Users/teseo/Downloads/Embryogram test/new/0in-analysis-07_09T15_10-analysis.hdf5"
path = "/Users/teseo/Downloads/Embryogram test/20250510_tracking-analysis-07_19T18_47-analysis.hdf5"

In [ ]:
hdf5_file = h5py.File(path, "r")
V, T = hdf5_file["mesh/v"][:].astype(float), hdf5_file["mesh/t"][:].astype(np.int32)

top = hdf5_file["bc/top"][:].astype(np.int32)
bottom = hdf5_file["bc/bottom"][:].astype(np.int32)
middle = hdf5_file["bc/middle"][:].astype(np.int32)

In [ ]:
centers = hdf5_file["bc_func/centers"][:].astype(float)

In [ ]:
nk = len(hdf5_file["bc_func"].keys())-2

disps = []

for i in range(nk):
    disps.append(hdf5_file[f"bc_func/disp{i+1}"][:].astype(float))

In [ ]:
# m=to_pyvista_mesh(V, T)
# # m=m.explode(0.5)
# plt = pv.Plotter()
# plt.add_mesh(m, show_edges=True, style='wireframe', line_width=1.0)
# plt.add_mesh(to_pyvista_mesh(V[top]), color='red', point_size=10, render_points_as_spheres=True, name='top')
# plt.add_mesh(to_pyvista_mesh(V[bottom]), color='green', point_size=10, render_points_as_spheres=True, name='bottom')
# plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=10, render_points_as_spheres=True, name='middle')
# plt.show()

In [ ]:
# m=m.explode(0.5)
plt = pv.Plotter()
plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=8, render_points_as_spheres=True, name='middle')
plt.add_mesh(to_pyvista_mesh(centers), color='red', point_size=7, render_points_as_spheres=True, name='middle')
plt.show()

In [ ]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line')
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1))



$$
P_0=(p_0^0|\dots|p_0^n)\qquad
P_1=(p_1^0|\dots|p_1^n)
$$

$$
\min_{M, t} E=\|P_0-MP_1 - t \mathbb{1}\|^2
$$

$$
E = P_0^T P_0
-2P_0^TMP_1
-2P_0^T t\mathbb{1} 
+ 2P_1^T M t\mathbb{1}
+P_1^TM^TMP_1 
+ \mathbb{1}^T t^T t \mathbb{1}
$$



$$
\nabla_M E = 
-2 P_0 P_1^T
+ 2P_1 \mathbb{1}^Tt^T
+2MP_1  P_1^T
$$

$$
\nabla_t E = 
-2P_0\mathbb{1}^T
+ 2M^T P_1  \mathbb{1}^T
+ 2t\mathbb{1} \mathbb{1}^T
$$

In [ ]:
import sympy as sp 


t = sp.MatrixSymbol('t', 3, 1)

X = sp.MatrixSymbol('x', 3, 1)
M = sp.MatrixSymbol('M', 3, 3)
PP = sp.MatrixSymbol('P', 3, 3)
K = sp.MatrixSymbol('K', 3, 1)



t1=X*t.T

t2 = M*PP


t3 = M.T * K
t3[0,0]


In [ ]:
def align(centers, disps, index):
    p0 = centers.T
    p1 = (centers + disps[index]).T
    one = np.ones((1, p0.shape[1]))

    n = one.shape[1]


    X = p1 @ one.T
    PP = p1 @ p1.T
    K = p1 @ one.T
    
    system = np.zeros((9+3, 9+3))
    rhs = np.zeros((9+3, ))

    rhs[:9] = (p0 @ p1.T).flatten()
    rhs[9:] = (p0 @ one.T).flatten()


    system[0,:3] = PP[:,0]
    system[1,:3] = PP[:,1]
    system[2,:3] = PP[:,2]

    system[3,3:6] = PP[:,0]
    system[4,3:6] = PP[:,1]
    system[5,3:6] = PP[:,2]

    system[6,6:9] = PP[:,0]
    system[7,6:9] = PP[:,1]
    system[8,6:9] = PP[:,2]


    system[0, 9] = X[0,0]
    system[1, 10] = X[0,0]
    system[2, 11] = X[0,0]

    system[0, 9] = X[1,0]
    system[1, 10] = X[1,0]
    system[2, 11] = X[1,0]

    system[0, 9] = X[2,0]
    system[1, 10] = X[2,0]
    system[2, 11] = X[2,0]

    system[9,:3] = K[:,0]
    system[9,3:6] = K[:,0]
    system[9,6:9] = K[:,0]

    system[9, 9] = n
    system[10, 10] = n
    system[11, 11] = n

    sol = np.linalg.solve(system, rhs)

    t = sol[9:12]

    return sol[:9].reshape((3, 3)), np.zeros(3,) #t.reshape((3, ))

M, t = align(centers, disps, 1)

print(M)
print(t)


In [ ]:
def icp(centers, d):
    p0 = centers
    p1 = centers + d
    mu0 = np.mean(p0, axis=0)
    mu1 = np.mean(p1, axis=0)

    p0 -= mu0
    p1 -= mu1
    t = mu0 - mu1

    cov = p1.T @ p0
    U, s, Ut = sp.linalg.svd(cov)
    R = Ut @ U.T

    return R, t

In [ ]:
def compute_align_disp(centers, disps, index):
    M, t = align(centers, disps, index)
    # M, t = icp(centers, disps[index])
    tmp = centers + disps[index]
    tmp1 = (M @ tmp.T).T + t

    return tmp1-centers

compute_align_disp(centers, disps, 1)

In [ ]:
new_disps = []
for i in range(len(disps)):
    new_disps.append(compute_align_disp(centers, disps, i))

In [ ]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line', color='red')

vertices = np.vstack([centers, centers + new_disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='linea', color='green')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line', color='red')

    vertices = np.vstack([centers, centers + new_disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='linea', color='green')
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1), continuous_update=False)





In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
f=206
plt.hist(np.linalg.norm(np.array(disps[f]-new_disps[f]), axis=1), bins=50)
plt.show()